# 套索肽 (Lasso Peptide) 分类器

胃肠道蛋白中套索肽的人工智能分类模型。

**流程**：FASTA 序列 → ESM-2 (冻结) → 1D Conv × 3 → Multi-Head Attention → MLP → 概率

**三种批量训练策略**：`multi_seed` · `cv` · `grid`

---

In [ ]:
# ============================================================
# 配置开关
# ============================================================
QUICK_MODE = True      # True: 少量数据快速测试; False: 完整训练
QUICK_SAMPLES = 100    # 速跑模式下每种样本取多少条
FORCE_REDOWNLOAD = False

# 默认使用 index 2 (ESM-2 35M)，改编号即可切换模型
SELECTED_MODEL_INDEX = 2  # 1=8M | 2=35M | 3=150M | 4=650M

In [ ]:
# ============================================================
# 1. 环境准备
# ============================================================
import os, sys, warnings, subprocess
from time import time
from copy import deepcopy
from itertools import product

import requests, shutil
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, auc, confusion_matrix, classification_report,
    precision_recall_curve,
)
from Bio import SeqIO
from transformers import EsmTokenizer, EsmModel, EsmConfig

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

print(f'Python:  {sys.version[:6]}')
print(f'Torch:   {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU:     {gpu.name}  ({gpu.total_memory / 1024**3:.1f} GB)')

---
## 第〇步：选择 ESM-2 模型

根据你的 GPU 显存选择最合适的预训练模型。ESM-2 在此仅做**冻结推理**（不微调），
所以实际显存需求低于训练所需。

In [ ]:
ESM_MODEL_CHOICES = [
    {"idx": 1, "name": "facebook/esm2_t6_8M_UR50D",     "label": "ESM-2 8M",   "params": "8M",   "dim": 320,  "vram": "≥ 2 GB",  "batch": 8},
    {"idx": 2, "name": "facebook/esm2_t12_35M_UR50D",   "label": "ESM-2 35M",  "params": "35M",  "dim": 480,  "vram": "≥ 4 GB",  "batch": 4},
    {"idx": 3, "name": "facebook/esm2_t30_150M_UR50D",  "label": "ESM-2 150M", "params": "150M", "dim": 640,  "vram": "≥ 8 GB",  "batch": 2},
    {"idx": 4, "name": "facebook/esm2_t33_650M_UR50D",  "label": "ESM-2 650M", "params": "650M", "dim": 1280, "vram": "≥ 16 GB", "batch": 1},
]

# 打印模型表
vram_gb = (torch.cuda.get_device_properties(0).total_memory / 1024**3) if torch.cuda.is_available() else 0
print("=" * 70)
print("  ESM-2 模型选择")
print("-" * 70)
print(f"  {'#':>3}  {'模型':<26} {'参数量':<8} {'dim':<6} {'推荐显存':<10} {'建议 batch':<10}")
print("-" * 70)
for m in ESM_MODEL_CHOICES:
    print(f"  {m['idx']:>3}  {m['label']:<26} {m['params']:<8} {m['dim']:<6} "
          f"{m['vram']:<10} {m['batch']:<10}")
print("-" * 70)
print(f"  当前 GPU 显存: {vram_gb:.1f} GB" if vram_gb else "  当前: CPU 模式")
print("=" * 70)

# 选定模型
selected = ESM_MODEL_CHOICES[SELECTED_MODEL_INDEX - 1]
ESM_MODEL = selected['name']
ESM_BATCH_SIZE = selected['batch']
embed_dim = selected['dim']
print(f'\n[*] 选定: {selected["label"]}  |  dim={embed_dim}  |  batch={ESM_BATCH_SIZE}')

In [ ]:
# ============================================================
# 2. 配置
# ============================================================
class Config:
    DATA_DIR = './lasso_data'
    DATASET_DIR = './lasso_dataset'
    CKPT_DIR = './lasso_checkpoints'
    RESULTS_DIR = './lasso_results'

    RAW_POS = os.path.join(DATA_DIR, 'raw_positives.fasta')
    RAW_NEG = os.path.join(DATA_DIR, 'raw_negatives.fasta')
    CLEAN_POS = os.path.join(DATA_DIR, 'clean_positives.fasta')
    CLEAN_NEG = os.path.join(DATA_DIR, 'clean_negatives.fasta')
    TRAIN_PT = os.path.join(DATASET_DIR, 'train.pt')
    VAL_PT = os.path.join(DATASET_DIR, 'val.pt')
    TEST_PT = os.path.join(DATASET_DIR, 'test.pt')
    BEST_CKPT = os.path.join(CKPT_DIR, 'best_model.pt')

    MAX_LEN = 100
    CD_HIT_THRESHOLD = 0.5

    BATCH_SIZE = 32
    EPOCHS = 10 if QUICK_MODE else 60
    LR = 1e-4
    WEIGHT_DECAY = 1e-5
    PATIENCE = 10
    DROPOUT = 0.3

    CNN_CHANNELS = [128, 128, 256]
    CNN_KERNELS = [5, 3, 3]
    ATTENTION_HEADS = 4
    MLP_HIDDEN = 64
    SEED = 42


def ensure_dirs(cfg):
    for d in [cfg.DATA_DIR, cfg.DATASET_DIR, cfg.CKPT_DIR, cfg.RESULTS_DIR]:
        os.makedirs(d, exist_ok=True)

cfg = Config()
ensure_dirs(cfg)
torch.manual_seed(cfg.SEED); np.random.seed(cfg.SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device: {device}  |  QUICK_MODE={QUICK_MODE}  |  Epochs={cfg.EPOCHS}')

---
## 第一步：数据获取

- **正样本**: [LassoPred](https://lassopred.accre.vanderbilt.edu/) 数据库的已知套索肽
- **负样本**: [UniProt Swiss-Prot](https://www.uniprot.org/) 中细菌非套索肽

In [ ]:
def fetch_lassopred(output_fasta):
    url = 'https://lassopred.accre.vanderbilt.edu/api/data/?page=1&size=4029'
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    r = requests.get(url, headers=headers)
    r.raise_for_status()
    data = r.json()['data']
    valid = 0
    with open(output_fasta, 'w') as f:
        for item in data:
            seq = item.get('Precursor_Sequence', '').strip()
            seq_id = item.get('LP_ID', '')
            if seq:
                f.write(f'>{seq_id}\n{seq}\n')
                valid += 1
    print(f'[+] Downloaded {valid} positive sequences → {output_fasta}')
    return valid

def fetch_uniprot_negatives(output_fasta, limit=20000):
    query = 'taxonomy_id:2 AND length:[40 TO 100] AND reviewed:true NOT family:"lasso peptide"'
    url = 'https://rest.uniprot.org/uniprotkb/stream'
    params = {'query': query, 'format': 'fasta', 'size': limit}
    r = requests.get(url, params=params, stream=True)
    r.raise_for_status()
    cnt = 0
    with open(output_fasta, 'w') as f:
        for line in r.iter_lines(decode_unicode=True):
            if not line: continue
            f.write(line + '\n')
            if line.startswith('>'): cnt += 1
            if cnt >= limit: break
    print(f'[+] Downloaded {cnt} negative sequences → {output_fasta}')
    return cnt

In [ ]:
t0 = time()
if not os.path.exists(cfg.RAW_POS) or FORCE_REDOWNLOAD:
    fetch_lassopred(cfg.RAW_POS)
else:
    n = sum(1 for _ in SeqIO.parse(cfg.RAW_POS, 'fasta'))
    print(f'[~] {cfg.RAW_POS} 已存在 ({n} 条)')
if not os.path.exists(cfg.RAW_NEG) or FORCE_REDOWNLOAD:
    limit = QUICK_SAMPLES * 2 if QUICK_MODE else 20000
    fetch_uniprot_negatives(cfg.RAW_NEG, limit=limit)
else:
    n = sum(1 for _ in SeqIO.parse(cfg.RAW_NEG, 'fasta'))
    print(f'[~] {cfg.RAW_NEG} 已存在 ({n} 条)')
print(f'  time: {time()-t0:.1f}s')

In [ ]:
# 序列长度分布
def get_lengths(fpath, max_n=None):
    lens = []
    for i, r in enumerate(SeqIO.parse(fpath, 'fasta')):
        if max_n and i >= max_n: break
        lens.append(len(r.seq))
    return lens

plt.figure(figsize=(10, 4))
n_lim = QUICK_SAMPLES if QUICK_MODE else None
pos_lens = get_lengths(cfg.RAW_POS, n_lim)
neg_lens = get_lengths(cfg.RAW_NEG, n_lim)
plt.hist(pos_lens, bins=40, alpha=0.6, color='firebrick', label='Positive (Lasso)', edgecolor='white')
plt.hist(neg_lens, bins=40, alpha=0.6, color='steelblue', label='Negative (Non-lasso)', edgecolor='white')
plt.xlabel('Sequence length (aa)'); plt.ylabel('Count')
plt.title('Sequence Length Distribution')
plt.legend(); plt.tight_layout(); plt.show()
print(f'Positive: {len(pos_lens)}  mean={np.mean(pos_lens):.1f} ± {np.std(pos_lens):.1f}')
print(f'Negative: {len(neg_lens)}  mean={np.mean(neg_lens):.1f} ± {np.std(neg_lens):.1f}')

---
## 第二步：CD-HIT 去冗余

In [ ]:
def run_cd_hit(in_fasta, out_fasta, threshold=0.5):
    if not os.path.exists(in_fasta):
        print(f'[!] file not found: {in_fasta}'); return
    if os.path.exists(out_fasta):
        n = sum(1 for _ in SeqIO.parse(out_fasta, 'fasta'))
        print(f'[~] CD-HIT cached: {out_fasta} ({n} seqs)'); return
    cmd = f'cd-hit -i {in_fasta} -o {out_fasta} -c {threshold} -n 3 -M 0 -d 0'
    try:
        subprocess.run(cmd, shell=True, check=True, stdout=subprocess.DEVNULL)
        nb = sum(1 for _ in SeqIO.parse(in_fasta, 'fasta'))
        na = sum(1 for _ in SeqIO.parse(out_fasta, 'fasta'))
        print(f'[+] CD-HIT: {nb} → {na} seqs')
    except subprocess.CalledProcessError:
        print(f'[!] CD-HIT not available — using raw data')
        shutil.copy(in_fasta, out_fasta)

if QUICK_MODE:
    for src, dst in [(cfg.RAW_POS, cfg.CLEAN_POS), (cfg.RAW_NEG, cfg.CLEAN_NEG)]:
        recs = list(SeqIO.parse(src, 'fasta'))[:QUICK_SAMPLES]
        SeqIO.write(recs, dst, 'fasta')
        print(f'[+] Quick: {len(recs)} seqs → {dst}')
else:
    run_cd_hit(cfg.RAW_POS, cfg.CLEAN_POS, cfg.CD_HIT_THRESHOLD)
    run_cd_hit(cfg.RAW_NEG, cfg.CLEAN_NEG, cfg.CD_HIT_THRESHOLD)

---
## 第三步：ESM-2 特征提取

ESM-2 冻结推理。输出 `(N, 100, embed_dim)` 保留序列维度。

In [ ]:
print(f'[*] Loading {ESM_MODEL} ...')
tokenizer = EsmTokenizer.from_pretrained(ESM_MODEL)
esm_model = EsmModel.from_pretrained(ESM_MODEL).to(device).eval()

In [ ]:
def extract_esm2_embeddings(fasta_file, batch_size, max_len):
    records = list(SeqIO.parse(fasta_file, 'fasta'))
    seqs = [str(r.seq) for r in records]
    ids = [r.id for r in records]
    all_e = []
    with torch.no_grad():
        for i in range(0, len(seqs), batch_size):
            batch = seqs[i:i + batch_size]
            inputs = tokenizer(batch, return_tensors='pt', padding='max_length',
                               max_length=max_len, truncation=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = esm_model(**inputs)
            hidden = outputs.last_hidden_state
            mask = inputs['attention_mask'].unsqueeze(-1).float()
            all_e.append((hidden * mask).cpu())
    return ids, torch.cat(all_e, dim=0)

In [ ]:
t0 = time()
pos_recs = list(SeqIO.parse(cfg.CLEAN_POS, 'fasta'))
neg_recs = list(SeqIO.parse(cfg.CLEAN_NEG, 'fasta'))

if len(neg_recs) > len(pos_recs) * 3:
    idx = torch.randperm(len(neg_recs))[:len(pos_recs) * 3]
    neg_recs = [neg_recs[i] for i in idx]
    print(f'[i] Downsampled negatives: {len(neg_recs)}')

tmp_p = os.path.join(cfg.DATA_DIR, '_p.fasta')
tmp_n = os.path.join(cfg.DATA_DIR, '_n.fasta')
SeqIO.write(pos_recs, tmp_p, 'fasta')
SeqIO.write(neg_recs, tmp_n, 'fasta')

pos_ids, pos_feat = extract_esm2_embeddings(tmp_p, ESM_BATCH_SIZE, cfg.MAX_LEN)
print(f'Pos: {pos_feat.shape}')
neg_ids, neg_feat = extract_esm2_embeddings(tmp_n, ESM_BATCH_SIZE, cfg.MAX_LEN)
print(f'Neg: {neg_feat.shape}  time: {time()-t0:.1f}s')
os.unlink(tmp_p); os.unlink(tmp_n)

pos_labels = torch.ones(pos_feat.size(0), 1)
neg_labels = torch.zeros(neg_feat.size(0), 1)

---
## 第四步：数据集划分

In [ ]:
X = torch.cat([pos_feat, neg_feat], dim=0)
y = torch.cat([pos_labels, neg_labels], dim=0)
all_ids = pos_ids + neg_ids

X_t, X_test, y_t, y_test, id_t, id_test = train_test_split(
    X, y, all_ids, test_size=0.1, random_state=cfg.SEED, stratify=y)
X_train, X_val, y_train, y_val, id_train, id_val = train_test_split(
    X_t, y_t, id_t, test_size=0.111, random_state=cfg.SEED, stratify=y_t)

for path, x, yy, ids_ in [
    (cfg.TRAIN_PT, X_train, y_train.squeeze(1), id_train),
    (cfg.VAL_PT, X_val, y_val.squeeze(1), id_val),
    (cfg.TEST_PT, X_test, y_test.squeeze(1), id_test)]:
    torch.save({'X': x, 'y': yy, 'ids': ids_}, path)

print(f'Train: {X_train.shape[0]}  Val: {X_val.shape[0]}  Test: {X_test.shape[0]}')
print(f'[+] Saved → {cfg.DATASET_DIR}/')

---
## 第五步：模型定义 (CNN + Attention + MLP)

In [ ]:
class LassoPeptideClassifier(nn.Module):
    def __init__(self, embed_dim=480, dropout=0.3):
        super().__init__()
        cnn_ch, cnn_ks = [128, 128, 256], [5, 3, 3]
        in_ch = embed_dim
        self.convs = nn.ModuleList()
        for ch, k in zip(cnn_ch, cnn_ks):
            self.convs.append(nn.Sequential(
                nn.Conv1d(in_ch, ch, k, padding=k // 2),
                nn.BatchNorm1d(ch), nn.ReLU(), nn.MaxPool1d(2)))
            in_ch = ch
        attn_dim = cnn_ch[-1]
        self.attn = nn.MultiheadAttention(attn_dim, 4, batch_first=True)
        self.attn_norm = nn.LayerNorm(attn_dim)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(attn_dim, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, x):
        x = x.permute(0, 2, 1)
        for c in self.convs: x = c(x)
        x = x.permute(0, 2, 1)
        a, _ = self.attn(x, x, x)
        x = self.attn_norm(x + a)
        return self.classifier(x.mean(dim=1))

model = LassoPeptideClassifier(embed_dim=embed_dim, dropout=cfg.DROPOUT).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params:,} params  |  dim={embed_dim}')

---
## 第六步：训练

In [ ]:
class LassoDataset(Dataset):
    def __init__(self, pt_path):
        data = torch.load(pt_path, map_location='cpu', weights_only=True)
        self.X, self.y = data['X'], data['y'].unsqueeze(1)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_set = LassoDataset(cfg.TRAIN_PT)
val_set = LassoDataset(cfg.VAL_PT)
test_set = LassoDataset(cfg.TEST_PT)

train_loader = DataLoader(train_set, batch_size=cfg.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=cfg.BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=cfg.BATCH_SIZE, shuffle=False)

pos_weight = ((train_set.y == 0).sum() / max((train_set.y == 1).sum(), 1)).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS)

In [ ]:
def compute_metrics(y_true, y_pred, y_prob):
    yt, yp, ypr = (t.cpu().numpy().ravel() for t in [y_true, y_pred, y_prob])
    nc = len(np.unique(yt))
    return {'loss': None, 'acc': accuracy_score(yt, yp),
            'prec': precision_score(yt, yp, zero_division=0),
            'rec': recall_score(yt, yp, zero_division=0),
            'f1': f1_score(yt, yp, zero_division=0),
            'auc': roc_auc_score(yt, ypr) if nc > 1 else float('nan')}

def train_one_epoch(m, ld, crit, opt):
    m.train(); tl = 0; Y, P, Pr = [], [], []
    for Xb, yb in ld:
        Xb, yb = Xb.to(device), yb.to(device)
        opt.zero_grad()
        logits = m(Xb); loss = crit(logits, yb); loss.backward(); opt.step()
        tl += loss.item() * Xb.size(0)
        prob = torch.sigmoid(logits.detach()).cpu(); pred = (prob >= .5).float()
        Y.append(yb.cpu()); P.append(pred); Pr.append(prob)
    met = compute_metrics(torch.cat(Y), torch.cat(P), torch.cat(Pr))
    met['loss'] = tl / len(ld.dataset); return met

@torch.no_grad()
def evaluate_model(m, ld, crit):
    m.eval(); tl = 0; Y, P, Pr = [], [], []
    for Xb, yb in ld:
        Xb, yb = Xb.to(device), yb.to(device)
        logits = m(Xb); loss = crit(logits, yb)
        tl += loss.item() * Xb.size(0)
        prob = torch.sigmoid(logits).cpu(); pred = (prob >= .5).float()
        Y.append(yb.cpu()); P.append(pred); Pr.append(prob)
    met = compute_metrics(torch.cat(Y), torch.cat(P), torch.cat(Pr))
    met['loss'] = tl / len(ld.dataset); return met

In [ ]:
def fmt(met):
    a = f"{met['auc']:.4f}" if not np.isnan(met['auc']) else ' N/A'
    return f"Loss:{met['loss']:.4f} Acc:{met['acc']:.3f} Prec:{met['prec']:.3f} Rec:{met['rec']:.3f} F1:{met['f1']:.3f} AUC:{a}"

history = {'tl': [], 'vf': [], 'va': []}
best_state = deepcopy(model.state_dict())
best_f1, best_ep, p_cnt = -1., 0, 0

t_train = time()
for ep in range(1, cfg.EPOCHS + 1):
    tm = train_one_epoch(model, train_loader, criterion, optimizer)
    vm = evaluate_model(model, val_loader, criterion)
    scheduler.step()
    history['tl'].append(tm['loss']); history['vf'].append(vm['f1']); history['va'].append(vm['auc'])
    print(f'E {ep:3d}/{cfg.EPOCHS}')
    print(f'      T: {fmt(tm)}'); print(f'      V: {fmt(vm)}')
    if vm['f1'] > best_f1:
        best_f1, best_ep = vm['f1'], ep
        best_state = deepcopy(model.state_dict()); p_cnt = 0
        print(f'      → best')
    else:
        p_cnt += 1
        if p_cnt >= cfg.PATIENCE: print(f'[ ] Early stop'); break
    print()

model.load_state_dict(best_state)
torch.save(best_state, cfg.BEST_CKPT)
print(f'='*50)
print(f'Done! time={time()-t_train:.0f}s  best=ep{best_ep} F1={best_f1:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(history['tl'], color='coral', lw=2); axes[0].set_title('Loss'); axes[0].grid(a=.3)
axes[1].plot(history['vf'], color='seagreen', lw=2)
axes[1].axhline(best_f1, c='gray', ls='--', label=f'B={best_f1:.4f}')
axes[1].set_title('Val F1'); axes[1].legend(); axes[1].grid(a=.3)
vd = [v for v in history['va'] if not np.isnan(v)]
if vd: axes[2].plot(history['va'], c='steelblue', lw=2)
else: axes[2].text(.5,.5,'N/A',ha='center',va='center',fontsize=20,c='gray')
axes[2].set_title('Val AUC'); axes[2].grid(a=.3)
plt.tight_layout(); plt.show()

---
## 第七步：测试集评估

In [ ]:
tm = evaluate_model(model, test_loader, criterion)
print(f'Test: {fmt(tm)}')

In [ ]:
@torch.no_grad()
def get_preds(m, ld):
    m.eval(); Y, P, Pr = [], [], []
    for Xb, yb in ld:
        logits = m(Xb.to(device)); prob = torch.sigmoid(logits).cpu()
        Y.append(yb); P.append((prob>=.5).float()); Pr.append(prob)
    return (torch.cat(Y).numpy().ravel(), torch.cat(P).numpy().ravel(), torch.cat(Pr).numpy().ravel())

y_true, y_pred, y_prob = get_preds(model, test_loader)

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(4, 3.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Neg','Pos'], yticklabels=['Neg','Pos'])
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title(f'Confusion Matrix (Acc={tm["acc"]:.3f})')
plt.tight_layout(); plt.show()

print(classification_report(y_true, y_pred, target_names=['Negative','Positive'], digits=4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
if len(np.unique(y_true)) > 1:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    axes[0].plot(fpr, tpr, lw=2, label=f'AUC={auc(fpr,tpr):.4f}')
    axes[0].plot([0,1],[0,1],'k--',a=.5); axes[0].legend(loc='lower right')
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    axes[1].plot(rec, prec, lw=2, c='darkorange')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].set_title('ROC Curve'); axes[0].grid(a=.3)
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].set_title('PR Curve'); axes[1].grid(a=.3)
axes[2].hist(y_prob[y_true==0], bins=30, a=.6, c='steelblue', label='Neg', ec='white')
axes[2].hist(y_prob[y_true==1], bins=30, a=.6, c='firebrick', label='Pos', ec='white')
axes[2].axvline(.5,c='gray',ls='--',label='Thr'); axes[2].set_xlabel('Probability'); axes[2].set_ylabel('Count')
axes[2].set_title('Distribution'); axes[2].legend(); axes[2].grid(a=.3)
plt.tight_layout(); plt.show()

---
## 第八步：预测新序列

In [ ]:
demo_seqs = [
    ('demo_lasso_1', 'MNPKKYSLAQEGVDNVKYLAGSSHGGAVIGTVKHVLEGDKTVTAEVKAKVTIVGNPNSGSSLIGGVTDTLNVGTGKGDVGVDVVK'),
    ('demo_lasso_2', 'MNKEFSLKQPEDAHKTYTQPKPHEWKCDDPKSYCCSWFCCDGYGHYQKICTYPK'),
    ('demo_random_1', 'MKKAVIVENKGLVSDELKANLIDDCRTGDVKVDLLARLDIGKVTKLAKKPGQTSKEFKEAVKAKLDEVKSRL'),
    ('demo_random_2', 'MALRVYADKKSVTIKDGTVTANGKPVKVIEGAKGSVVTGKVEMTLGDDLVIATVKGDKAVVVRGSVGDAVAA'),
    ('insulin', 'MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN'),
]

demo_fasta = os.path.join(cfg.DATA_DIR, 'demo.fasta')
with open(demo_fasta, 'w') as f:
    for sid, s in demo_seqs: f.write(f'>{sid}\n{s}\n')

records = list(SeqIO.parse(demo_fasta, 'fasta'))
seq_ids = [r.id for r in records]
_, embeds = extract_esm2_embeddings(demo_fasta, ESM_BATCH_SIZE, cfg.MAX_LEN)

model.eval()
with torch.no_grad():
    logits = model(embeds.to(device))
    probs = torch.sigmoid(logits).squeeze(-1).cpu().numpy()

df = pd.DataFrame({'sequence_id': seq_ids, 'probability': probs,
                   'prediction': ['positive' if p >= .5 else 'negative' for p in probs]})
print(df.to_string(index=False))

In [ ]:
colors = ['firebrick' if p >= .5 else 'steelblue' for p in df['probability']]
plt.figure(figsize=(7,4))
plt.bar(df['sequence_id'], df['probability'], color=colors, ec='white')
plt.axhline(.5,c='gray',ls='--',label='Threshold')
plt.ylim(0,1.05); plt.ylabel('Lasso Peptide Probability')
plt.title('Prediction Results'); plt.xticks(rotation=15)
plt.legend(); plt.tight_layout(); plt.show()

---
## 第九步：批量训练实验

三种策略验证模型稳定性：
- **multi_seed**: 多随机种子找最佳
- **cv**: K 折交叉验证
- **grid**: 超参数网格搜索

> 该 Cell 可独立运行（前提是已完成前 7 步的特征提取和 .pt 文件生成）

### 9.1 多随机种子训练

In [ ]:
N_RUNS = 3  # 快速模式下建议 ≤ 3
BASE_SEED = 42

best_f1_multi = -1.; best_seed = 0
all_f1s, all_aucs = [], []

t0 = time()
for i in range(N_RUNS):
    seed = BASE_SEED + i
    torch.manual_seed(seed); np.random.seed(seed)
    
    m = LassoPeptideClassifier(embed_dim=embed_dim, dropout=cfg.DROPOUT).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.EPOCHS)
    pw = ((train_set.y == 0).sum() / max((train_set.y == 1).sum(), 1)).to(device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    
    best_vf = -1.; p_cnt = 0; best_st = None
    for ep in range(1, cfg.EPOCHS + 1):
        train_one_epoch(m, train_loader, crit, opt)
        vm = evaluate_model(m, val_loader, crit)
        sch.step()
        vf = vm['f1'] if not np.isnan(vm['f1']) else -1.
        if vf > best_vf:
            best_vf = vf; best_st = deepcopy(m.state_dict()); p_cnt = 0
        else:
            p_cnt += 1
            if p_cnt >= cfg.PATIENCE: break
    
    if best_st: m.load_state_dict(best_st)
    tm = evaluate_model(m, test_loader, crit)
    f1 = tm['f1'] if not np.isnan(tm['f1']) else 0.
    all_f1s.append(f1)
    all_aucs.append(tm['auc'] if not np.isnan(tm['auc']) else float('nan'))
    
    if f1 > best_f1_multi:
        best_f1_multi = f1; best_seed = seed
        torch.save(m.state_dict(), cfg.BEST_CKPT)
    print(f'[Run {i+1}] seed={seed:3d}  Test F1={f1:.4f}')

af1 = np.array(all_f1s)
aa = np.array([a for a in all_aucs if not np.isnan(a)])
print(f'\nMulti-seed ({N_RUNS} runs):')
print(f'  Test F1:  {af1.mean():.4f} ± {af1.std():.4f}')
if len(aa) > 0:
    print(f'  Test AUC: {aa.mean():.4f} ± {aa.std():.4f}')
print(f'  Best: seed={best_seed} F1={best_f1_multi:.4f}')
print(f'  Time: {time()-t0:.0f}s')

### 9.2 K 折交叉验证

In [ ]:
class CombinedDataset(Dataset):
    def __init__(self, pt_a, pt_b):
        da = torch.load(pt_a, map_location='cpu', weights_only=True)
        db = torch.load(pt_b, map_location='cpu', weights_only=True)
        self.X = torch.cat([da['X'], db['X']], dim=0)
        self.y = torch.cat([da['y'].unsqueeze(1), db['y'].unsqueeze(1)], dim=0)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

N_FOLDS = 3  # 快速模式下建议 ≤ 3
combined = CombinedDataset(cfg.TRAIN_PT, cfg.VAL_PT)
kfold = KFold(n_splits=N_FOLDS, shuffle=True, random_state=cfg.SEED)

cv_results = []
t0 = time()
for fi, (tr_idx, va_idx) in enumerate(kfold.split(range(len(combined)))):
    tr_ld = DataLoader(combined, batch_size=cfg.BATCH_SIZE, sampler=SubsetRandomSampler(tr_idx))
    va_ld = DataLoader(combined, batch_size=cfg.BATCH_SIZE, sampler=SubsetRandomSampler(va_idx))
    
    m = LassoPeptideClassifier(embed_dim=embed_dim, dropout=cfg.DROPOUT).to(device)
    pc = (combined.y[tr_idx] == 1).sum().item()
    pw = torch.tensor([(len(tr_idx) - pc) / max(pc, 1)]).to(device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    opt = torch.optim.AdamW(m.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.EPOCHS)
    
    bvf = -1.; bst = None; pcnt = 0
    for ep in range(1, cfg.EPOCHS + 1):
        train_one_epoch(m, tr_ld, crit, opt)
        vm = evaluate_model(m, va_ld, crit)
        sch.step()
        vf = vm['f1'] if not np.isnan(vm['f1']) else -1.
        if vf > bvf: bvf = vf; bst = deepcopy(m.state_dict()); pcnt = 0
        else:
            pcnt += 1
            if pcnt >= cfg.PATIENCE: break
    
    if bst: m.load_state_dict(bst)
    tm = evaluate_model(m, test_loader, crit)
    f1 = tm['f1'] if not np.isnan(tm['f1']) else 0.
    cv_results.append(f1)
    print(f'[Fold {fi+1}] Test F1: {f1:.4f}')

cv_arr = np.array(cv_results)
print(f'\nCV Summary ({N_FOLDS} folds):')
print(f'  Test F1: {cv_arr.mean():.4f} ± {cv_arr.std():.4f}')
print(f'  Time: {time()-t0:.0f}s')

### 9.3 超参数网格搜索

In [ ]:
# 快速模式下用小网格
if QUICK_MODE:
    grid_lr = [1e-4, 5e-5]
    grid_dp = [0.3, 0.5]
    grid_ch = [[128, 128, 256]]
    grid_bs = [32]
else:
    grid_lr = [1e-3, 1e-4, 5e-5]
    grid_dp = [0.2, 0.3, 0.5]
    grid_ch = [[64, 128, 256], [128, 128, 256], [128, 256, 512]]
    grid_bs = [16, 32]

combos = list(product(grid_lr, grid_dp, grid_ch, grid_bs))
print(f'Grid: {len(combos)} combinations')
grid_res = []
t0 = time()

for ci, (lr, dp, ch, bs) in enumerate(combos):
    torch.manual_seed(cfg.SEED)
    tr_ld = DataLoader(train_set, batch_size=bs, shuffle=True)
    
    m = LassoPeptideClassifier(embed_dim=embed_dim, dropout=dp).to(device)
    m.convs = nn.ModuleList()
    in_ch = embed_dim
    for c, k in zip(ch, [5, 3, 3]):
        m.convs.append(nn.Sequential(
            nn.Conv1d(in_ch, c, k, padding=k//2),
            nn.BatchNorm1d(c), nn.ReLU(), nn.MaxPool1d(2)))
        in_ch = c
    
    pw = ((train_set.y == 0).sum() / max((train_set.y == 1).sum(), 1)).to(device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=cfg.WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.EPOCHS)
    
    bvf = -1.; best_st = None; pcnt = 0
    for ep in range(1, cfg.EPOCHS + 1):
        train_one_epoch(m, tr_ld, crit, opt)
        vm = evaluate_model(m, val_loader, crit)
        sch.step()
        vf = vm['f1'] if not np.isnan(vm['f1']) else -1.
        if vf > bvf: bvf = vf; best_st = deepcopy(m.state_dict()); pcnt = 0
        else:
            pcnt += 1
            if pcnt >= cfg.PATIENCE: break
    
    if best_st: m.load_state_dict(best_st)
    tm = evaluate_model(m, test_loader, crit)
    f1 = tm['f1'] if not np.isnan(tm['f1']) else 0.
    grid_res.append({'lr': lr, 'dp': dp, 'ch': ch, 'bs': bs, 'f1': f1})
    print(f'[{ci+1}/{len(combos)}] lr={lr:.0e} dp={dp} ch={ch} bs={bs}  |  F1={f1:.4f}')

grid_res.sort(key=lambda x: x['f1'], reverse=True)
print(f'\nTop-3:')
for r in grid_res[:3]:
    print(f'  F1={r["f1"]:.4f}  lr={r["lr"]:.0e} dp={r["dp"]} ch={r["ch"]} bs={r["bs"]}')
print(f'  Time: {time()-t0:.0f}s')

---
## 总结

| 产出 | 路径 |
|------|------|
| 模型权重 | `lasso_checkpoints/best_model.pt` |
| 特征数据集 | `lasso_dataset/train.pt / val.pt / test.pt` |
| 评估图表 | `lasso_results/` |
| 一键批量实验 | `python run_experiment.py --strategy multi_seed --runs 5` |

In [ ]:
print('=== Summary ===')
print(f'ESM:       {selected["label"]} ({selected["params"]}, dim={embed_dim})')
print(f'Params:    {n_params:,}')
print(f'Train:     {len(train_set)}  |  Val: {len(val_set)}  |  Test: {len(test_set)}')
print(f'Test F1:   {tm["f1"]:.4f}')
a = tm['auc']; print(f'Test AUC:  {a:.4f}' if not np.isnan(a) else 'Test AUC:  N/A')
print(f'Best epoch:{best_ep}  |  Time: {time()-t_train:.0f}s')
print()
print('Batch experiment:')
print(f'  python run_experiment.py --strategy multi_seed --runs 5 --esm-model {ESM_MODEL}')